# Atelier pandas : de vos tableaux Excel à un tableau de bord

Bienvenue ! Ici, **rien à installer** : tout se passe dans votre navigateur, et vos données ne quittent pas votre ordinateur.

**Comment ça marche ?**
1. Cliquez sur une case grise (une « cellule de code »).
2. Appuyez sur **Maj + Entrée** (ou sur le bouton ▶ en haut). Le résultat s'affiche dessous.
3. Passez à la case suivante et recommencez.

La première exécution est un peu longue (une trentaine de secondes) : le navigateur prépare Python. C'est normal.

> Vous ne pouvez rien casser. En cas de souci : menu **Noyau (Kernel) → Redémarrer**, puis reprenez depuis le début.

## 1. Charger un tableau
On charge un fichier CSV, l'équivalent d'un Excel « à plat ». Ce tableau s'appelle un **DataFrame** (souvent abrégé `df`).

In [ ]:
import pandas as pd

df = pd.read_csv("donnees/batiments_exemple.csv", sep=";")
df.head()      # affiche les 5 premières lignes

*Les données de cet atelier sont fictives, dans le style des données ouvertes de la Ville. Pour utiliser un vrai fichier, déposez-le dans le dossier `donnees` (panneau de gauche) et changez le nom entre guillemets.*

## 2. Regarder ce qu'on a

In [ ]:
print("Nombre de lignes et de colonnes :", df.shape)
df.info()      # colonnes, types, valeurs manquantes

In [ ]:
df.describe()   # statistiques rapides : moyenne, min, max...

## 3. Nettoyer
Sur la colonne `volume_m3`, quelques cases sont vides. On les repère, puis on retire ces lignes.

In [ ]:
print("Cases vides dans volume_m3 :", df["volume_m3"].isna().sum())
df = df.dropna(subset=["volume_m3"])
print("Il reste", len(df), "lignes")

## 4. Filtrer
Comme le filtre d'Excel : on garde seulement les lignes qui répondent à une condition.

In [ ]:
bureaux = df[df["type_batiment"] == "Bureau"]
bureaux.head()

**À vous de jouer :** remplacez `"Bureau"` par `"Commercial"`, puis ré-exécutez la case (Maj + Entrée).

## 5. Regrouper (« regrouper par »)
Combien de m³ par arrondissement ? C'est le principe du sous-total.

In [ ]:
volumes = df.groupby("arrondissement")["volume_m3"].sum().sort_values(ascending=False)
volumes.head(5)     # les 5 plus gros

## 6. Le tableau croisé dynamique (TCD)
Exactement ce que vous faites dans Excel en glissant des champs dans *Lignes*, *Colonnes* et *Valeurs*.

In [ ]:
tcd = pd.pivot_table(
    df,
    index="arrondissement",      # les lignes
    columns="type_batiment",     # les colonnes
    values="volume_m3",          # ce qu'on additionne
    aggfunc="sum",               # somme (essayez "mean" pour la moyenne)
    fill_value=0,                # 0 à la place des cases vides
    margins=True,                # ajoute les totaux
    margins_name="Total",
)
tcd

## 7. Un graphique
Une ligne suffit.

In [ ]:
import matplotlib.pyplot as plt

tcd.drop(index="Total", columns="Total").plot(kind="bar", stacked=True, figsize=(11, 5),
                                               title="Volumes bâtis par arrondissement")
plt.ylabel("m³")
plt.show()

**À vous de jouer :** dans la case du TCD, remplacez `"type_batiment"` par `"annee"`, ré-exécutez le TCD, puis le graphique.

## 8. Récupérer votre résultat
On enregistre le TCD en Excel, puis un lien de téléchargement apparaît.

In [ ]:
try:                                # spécifique au navigateur : charge l'outil Excel
    import piplite
    await piplite.install("openpyxl")
except ImportError:
    pass
import openpyxl                    # nécessaire pour écrire un fichier Excel
import base64
from IPython.display import HTML, display

def telecharger(nom, ouvrir=False):
    """Affiche un lien pour enregistrer le fichier sur votre ordinateur
    (et, si ouvrir=True, un second lien pour l'ouvrir dans un nouvel onglet)."""
    contenu = open(nom, "rb").read()
    try:                                    # dans le navigateur
        import js
        from pyodide.ffi import to_js
        blob = js.Blob.new(to_js([to_js(contenu)]),
                           to_js({"type": "application/octet-stream"}, dict_converter=js.Object.fromEntries))
        adresse = js.URL.createObjectURL(blob)
    except ImportError:                     # Python classique (hors navigateur)
        adresse = "data:application/octet-stream;base64," + base64.b64encode(contenu).decode()
    liens = (f'<a download="{nom}" href="{adresse}" style="font-size:1.1em">'
             f'⬇ Cliquez ici pour télécharger {nom}</a>')
    if ouvrir:
        liens += (f' &nbsp;&nbsp; <a target="_blank" href="{adresse}" style="font-size:1.1em">'
                  f'Ouvrir dans un nouvel onglet</a>')
    display(HTML(liens))

tcd.to_excel("mon_tcd.xlsx", sheet_name="TCD")
telecharger("mon_tcd.xlsx")

Si le lien ne fonctionne pas : dans le panneau de gauche, clic droit sur `mon_tcd.xlsx` puis **Télécharger**.

**Bravo, vous venez de refaire un TCD Excel avec du code.**

## 9. Des graphiques interactifs (Plotly)
Les graphiques précédents sont des images. Avec **Plotly**, on peut survoler, zoomer et filtrer avec la souris.

D'abord on charge Plotly (quelques secondes, uniquement la première fois).

In [ ]:
try:                                # spécifique au navigateur
    import piplite
    await piplite.install("plotly")
except ImportError:
    pass

import plotly.express as px
print("Plotly est prêt")

In [ ]:
ordre = ["1er"] + [f"{i}e" for i in range(2, 21)]

fig_arr = px.bar(
    df.groupby(["arrondissement", "type_batiment"], as_index=False)["volume_m3"].sum(),
    x="arrondissement", y="volume_m3", color="type_batiment",
    category_orders={"arrondissement": ordre},
    title="Volumes bâtis par arrondissement et par type",
)

fig_annee = px.line(
    df.groupby("annee", as_index=False)["volume_m3"].sum(),
    x="annee", y="volume_m3", markers=True,
    title="Évolution des volumes par année",
)

fig_type = px.pie(df, names="type_batiment", values="volume_m3",
                  title="Répartition des volumes par type de bâtiment")

**À vous de jouer :** dans `fig_arr`, remplacez `px.bar` par `px.line` (ou `px.area`) et ré-exécutez.

## 10. Télécharger votre tableau de bord (une page web autonome)
La cellule ci-dessous assemble vos graphiques dans **une seule page web** : le texte (HTML), la mise en forme (CSS) et le moteur des graphiques (JavaScript Plotly) sont tous dans le même fichier.

Vous pouvez l'envoyer par mail ou le double-cliquer : il s'ouvre dans n'importe quel navigateur, **sans internet et sans Python**.

In [ ]:
import html
from plotly.offline import get_plotlyjs

STYLE = """
:root { --papier:#EEF2F5; --encre:#142434; --trait:#C3CED8; --bleu:#1F5A99; --piquet:#E0A81C; }
* { box-sizing: border-box; }
body { margin:0; background:var(--papier); color:var(--encre); font:16px/1.5 "Segoe UI", Arial, sans-serif; }
header { padding:32px 5vw 20px; border-bottom:1px solid var(--trait); border-left:8px solid var(--piquet); }
h1 { margin:0; font:600 2rem/1.1 "Bahnschrift","Segoe UI",Arial,sans-serif; }
main { display:grid; grid-template-columns:repeat(auto-fit,minmax(min(100%,520px),1fr)); gap:24px; padding:24px 5vw 48px; }
.planche { background:#fff; border:1px solid var(--trait); padding:8px; min-width:0; }
"""

def exporter_dashboard(graphiques, titre="Mon tableau de bord", fichier="mon_dashboard.html"):
    """graphiques = liste de graphiques Plotly. Écrit une page web autonome."""
    blocs = "\n".join(
        '<section class="planche">' + g.to_html(full_html=False, include_plotlyjs=False,
                                                config={"responsive": True}) + "</section>"
        for g in graphiques)
    page = (f'<!DOCTYPE html><html lang="fr"><head><meta charset="utf-8">'
            f'<meta name="viewport" content="width=device-width, initial-scale=1">'
            f'<title>{html.escape(titre)}</title><style>{STYLE}</style>'
            f'<script>{get_plotlyjs()}</script></head>'
            f'<body><header><h1>{html.escape(titre)}</h1></header><main>{blocs}</main></body></html>')
    with open(fichier, "w", encoding="utf-8") as f:
        f.write(page)
    return fichier

fichier = exporter_dashboard([fig_arr, fig_annee, fig_type], titre="Volumes bâtis (atelier pandas)")
telecharger(fichier, ouvrir=True)

Un peu lent ? C'est normal : la page contient tout le moteur des graphiques (environ 5 Mo).

Si le lien ne réagit pas : panneau de gauche, clic droit sur `mon_dashboard.html` puis **Télécharger**.

**Vous avez maintenant un vrai tableau de bord web, créé par vos soins.**

---
# Partie B — passer à de vraies données (open data)

Jusqu'ici, on a travaillé sur `batiments_exemple.csv`, un fichier fictif fourni avec l'atelier.
Cette partie montre comment brancher le même code sur de **vraies données ouvertes** de la Ville de Paris,
sans avoir à les télécharger à la main.

Deux façons d'arriver au même résultat, à ne pas confondre :

| | **Ici, dans le navigateur (JupyterLite)** | **Chez vous, en local (installation Python)** |
|---|---|---|
| Comment on va chercher la donnée | on **interroge une API** (adresse web qui répond en JSON) | pareil, ou on **télécharge un CSV** une fois pour toutes |
| Pourquoi cette différence | le navigateur applique des règles de sécurité (CORS) qui bloquent parfois les téléchargements de fichiers volumineux | aucune de ces restrictions : on peut tout faire |
| Fichier de secours | `donnees/batiments_exemple.csv`, toujours disponible | idem, ou son propre CSV téléchargé une fois |

## 9. Pourquoi une API plutôt qu'un simple fichier ?

Un fichier CSV est une **photo** des données à un instant donné : pour le mettre à jour, il faut le retélécharger à la main.

Une **API** (Interface de Programmation), c'est une adresse web à laquelle on pose une question précise, et qui répond en JSON
(un format texte structuré, proche d'un dictionnaire Python). On peut lui demander : *« donne-moi seulement les bâtiments du
5e arrondissement »*, et elle ne renvoie que ça, sans qu'on ait à tout télécharger puis filtrer soi-même.

La Ville de Paris met à disposition son [portail open data](https://opendata.paris.fr) sur ce principe. Presque tous ses
jeux de données sont accessibles par la même API, appelée **Explore API v2.1**. Le jeu de données `les-arbres` (l'inventaire
des arbres de Paris) sert d'exemple ci-dessous : il est libre d'accès, sans clé ni inscription, ce qui en fait un bon
premier contact avec une API.

**Pourquoi celui-ci et pas un autre ?** Il est mis à jour régulièrement, ne contient aucune donnée sensible, et sa taille
(plus de 200 000 lignes) illustre bien l'intérêt de *filtrer côté serveur* plutôt que de tout rapatrier.

## 10. Récupérer les données depuis le notebook (ici, dans le navigateur)

Dans le navigateur, on ne peut pas utiliser la bibliothèque `requests` habituelle : Pyodide fournit `pyfetch`, l'équivalent
adapté au navigateur. Le principe reste le même : on construit une adresse web, on l'appelle, on lit la réponse JSON.

In [ ]:
async def recuperer_arbres(arrondissement="PARIS 5E ARRDT", nb_lignes=300):
    """Interroge l'API Explore de la Ville de Paris et renvoie un DataFrame.
    Fonctionne dans le navigateur (Pyodide) et échoue proprement sinon."""
    import urllib.parse
    base = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/les-arbres/records"
    parametres = {
        "limit": nb_lignes,                                   # nombre de lignes demandées
        "where": f'arrondissement="{arrondissement}"',        # filtre : posé au serveur, pas chez nous
        "select": "idbase,libellefrancais,genre,espece,hauteurenm,circonferenceencm,arrondissement",
    }
    adresse = base + "?" + urllib.parse.urlencode(parametres)

    try:
        from pyodide.http import pyfetch
        reponse = await pyfetch(adresse)
        if not reponse.ok:
            raise RuntimeError(f"L'API a répondu {reponse.status}")
        donnees = await reponse.json()
        return pd.json_normalize(donnees["results"])
    except ImportError:
        raise RuntimeError("Cette fonction est prévue pour le navigateur (Pyodide). "
                            "En local, utilisez recuperer_arbres_local() (cellule suivante).")

arbres = await recuperer_arbres()
print(len(arbres), "arbres récupérés")
arbres.head()

**Si la cellule échoue** (message d'erreur réseau, "Failed to fetch", ou `CORS`) : le poste est probablement derrière un
filtre qui bloque `opendata.paris.fr`. Ce n'est pas un problème de code — testez plutôt la version « en local » ci-dessous,
ou repliez-vous sur `donnees/batiments_exemple.csv`.

**À vous de jouer :** changez `"PARIS 5E ARRDT"` par votre arrondissement, ré-exécutez.

## 11. La même chose, en local (hors navigateur)

Si vous travaillez plus tard sur votre ordinateur, dans un vrai Python installé, **on ne peut pas utiliser `pyfetch`** :
il n'existe que dans le navigateur. On utilise la bibliothèque `requests`, standard en Python. Voici l'équivalent — cette
cellule ne s'exécute pas ici (Pyodide n'a pas `requests` réseau), mais vous pouvez la copier telle quelle sur votre poste.

In [ ]:
# À exécuter en LOCAL uniquement (pas dans ce notebook navigateur) :
#
# import requests
# import pandas as pd
#
# def recuperer_arbres_local(arrondissement="PARIS 5E ARRDT", nb_lignes=300):
#     base = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/les-arbres/records"
#     parametres = {
#         "limit": nb_lignes,
#         "where": f'arrondissement="{arrondissement}"',
#         "select": "idbase,libellefrancais,genre,espece,hauteurenm,circonferenceencm,arrondissement",
#     }
#     reponse = requests.get(base, params=parametres, timeout=15)
#     reponse.raise_for_status()          # lève une erreur claire si l'API répond mal
#     return pd.json_normalize(reponse.json()["results"])
#
# arbres = recuperer_arbres_local()
#
# # Pour ne pas relancer l'appel API à chaque fois : on enregistre une copie locale.
# arbres.to_csv("donnees/arbres_5e.csv", sep=";", index=False)
# # La prochaine fois : arbres = pd.read_csv("donnees/arbres_5e.csv", sep=";")

**Pourquoi enregistrer une copie (`to_csv`) après l'appel ?** Pour ne pas re-interroger l'API à chaque exécution du
notebook : c'est plus rapide, ça fonctionne même sans connexion, et ça évite de solliciter le serveur de la Ville
pour rien. C'est le même réflexe que `donnees/batiments_exemple.csv` : une donnée qu'on interroge une fois, puis qu'on
rejoue localement.

## 12. Installer Python sur son propre poste (pour aller plus loin que le navigateur)

Ce que vous faites ici dans JupyterLite tient dans le navigateur, mais un Python installé en local permet en plus :
la connexion à de vraies bases de données, les gros fichiers, et de travailler sans connexion. Si vous voulez essayer
chez vous, une seule fois, sur votre propre poste :

1. **Installer Python.** Téléchargez-le sur [python.org](https://www.python.org/downloads/) (version 3.11 ou plus récente).
   Sur l'écran d'installation, cochez bien **"Add python.exe to PATH"**.
2. **Ouvrir une invite de commandes** (`cmd` ou PowerShell) et vérifier : `python --version`.
3. **Installer les bibliothèques du cours**, en une seule ligne :
   `pip install pandas numpy matplotlib plotly openpyxl jupyterlab requests`
4. **Lancer JupyterLab**, l'interface qui ressemble à celle-ci : `jupyter lab`. Une page s'ouvre dans le navigateur,
   mais cette fois Python tourne réellement sur votre machine, pas dans le navigateur.
5. Déposez `cours_pandas.ipynb` et le dossier `donnees` dans le même dossier que celui d'où vous lancez `jupyter lab`,
   puis ouvrez le notebook : tout le reste du cours fonctionne à l'identique.

**Pas de droits administrateur nécessaires** si vous installez Python "pour mon compte uniquement" (option proposée à
l'étape 1). Si l'installation est bloquée par la politique du poste, il faudra la demander à votre support informatique.

---
# Partie C — pour aller plus loin

## 13. Les autres façons de faire un TCD en pandas

On a vu `pivot_table`. Ce n'est pas la seule fonction : selon ce qu'on veut, une autre est parfois plus simple.

| Fonction | À utiliser quand... | Limite |
|---|---|---|
| `pivot_table` | on veut **additionner, moyenner...** des valeurs (le cas le plus courant) | rien de particulier : c'est la plus complète |
| `crosstab` | on veut juste **compter des occurrences** (combien de lignes dans chaque croisement) | pas de colonne de valeurs à agréger, seulement des comptages |
| `pivot` | on a déjà **une seule valeur par croisement** (pas de doublon à additionner) | plante s'il y a plusieurs lignes pour un même croisement |
| `groupby().unstack()` | on part d'un `groupby` déjà écrit, et on veut juste l'afficher "à plat" | un peu plus de code, mais très flexible |

Exemples, sur le même jeu de données :

In [ ]:
# 1) crosstab : combien de bâtiments (lignes) par arrondissement et par type — sans les additionner
pd.crosstab(df["arrondissement"], df["type_batiment"])

In [ ]:
# 2) groupby + unstack : la même somme que pivot_table, en partant d'un groupby
(df.groupby(["arrondissement", "type_batiment"])["volume_m3"]
   .sum()
   .unstack(fill_value=0))

**Pourquoi utiliser `pivot_table` par défaut ?** C'est la seule des trois qui gère nativement plusieurs types
d'agrégation à la fois (`aggfunc=["sum", "mean"]`), les totaux (`margins=True`) et les valeurs manquantes
(`fill_value`) en un seul appel. Les deux autres sont utiles quand on sait précisément qu'on n'a pas besoin de ça.

## 14. Pourquoi notre export Excel n'est pas un vrai "TCD Excel"

Quand vous ouvrez `mon_tcd.xlsx`, vous voyez des chiffres, mais pas le panneau "Champs de tableau croisé dynamique" à
droite, ni le menu Analyse/Création dans le ruban Excel. C'est normal, et voici pourquoi.

Un vrai tableau croisé dynamique Excel n'est pas juste un tableau de chiffres : c'est un **objet interactif**, avec :
- une **copie brute des données sources**, stockée dans le fichier (le "cache du TCD") ;
- une **définition XML** de la disposition (quels champs en ligne, en colonne, en valeur) ;
- un moteur de recalcul propre à Excel, qui régénère l'affichage quand on glisse un champ.

`df.to_excel()` (utilisé au cours 8) n'écrit que le **résultat déjà calculé** : des cellules avec des chiffres, sans
ce mécanisme derrière. C'est une photo, pas l'appareil qui l'a prise.

**Pourquoi ne pas générer ce vrai objet directement en Python ?** Une bibliothèque existe, `xlsxwriter`, avec un
module dédié (`add_pivot_table`… en réalité, même elle ne le propose pas nativement). En pratique, écrire un cache de
TCD valide "à la main" demande de reproduire un format binaire interne d'Excel, peu documenté, différent selon les
versions d'Excel, et qui casse silencieusement si un détail est mal formé. Le rapport effort/fiabilité n'est pas bon
pour un cours d'initiation.

**Ce que je vous recommande à la place**, selon ce que vous voulez :
- **Analyser en Python** : on reste sur `pivot_table`, exporté en `.xlsx` comme au cours 8. C'est suffisant pour lire un résultat.
- **Manipuler un vrai TCD dans Excel** : exportez les **données brutes** (`df.to_excel("donnees_brutes.xlsx")`, sans
  passer par `pivot_table`), puis dans Excel : **Insertion → Tableau croisé dynamique**. C'est le même geste que
  d'habitude, sur des données préparées par Python en amont.

In [ ]:
# Pour la seconde option : exporter les données prêtes à l'emploi, à glisser-déposer soi-même dans Excel
df.to_excel("donnees_brutes.xlsx", index=False, sheet_name="Données")
telecharger("donnees_brutes.xlsx")

## 15. Tour d'horizon : quel graphique pour quelle question ?

Plotly propose une trentaine de types de graphiques. En voici les plus utiles pour un usage courant, avec la question
à laquelle chacun répond. Tous se créent de la même manière : `px.<type>(df, x=..., y=..., ...)`.

| Fonction | Répond à la question... | Exemple d'usage |
|---|---|---|
| `px.bar` | Comparer des quantités entre catégories | volume par arrondissement |
| `px.line` | Voir une évolution dans le temps | volume par année |
| `px.area` | Une évolution, en insistant sur le cumul | volume cumulé par année |
| `px.scatter` | Deux variables numériques sont-elles liées ? | surface vs volume |
| `px.pie` | Une répartition en proportions (peu de catégories) | part de chaque type de bâtiment |
| `px.histogram` | Comment se distribue une variable numérique ? | distribution des volumes |
| `px.box` | Comparer des distributions entre groupes (médiane, écart) | volumes par type, avec les valeurs atypiques |
| `px.violin` | Comme `box`, avec la forme complète de la distribution | idem, en plus détaillé |
| `px.imshow` | Une matrice de valeurs, en couleur (carte de chaleur) | le TCD lui-même, visualisé en couleurs |
| `px.treemap` | Une hiérarchie de proportions imbriquées | volume par arrondissement puis par type |
| `px.sunburst` | Comme `treemap`, en cercles concentriques | la même hiérarchie, présentation différente |
| `px.scatter_mapbox` | Positionner des points sur une carte | localiser des bâtiments (latitude/longitude) |

**Comment choisir ?** Trois questions à se poser dans l'ordre :
1. *Je compare des catégories, ou je regarde une évolution dans le temps ?* → `bar` ou `line`.
2. *Je regarde une seule variable, ou une relation entre deux ?* → `histogram`/`box` pour une seule, `scatter` pour deux.
3. *J'ai une hiérarchie (catégorie dans une catégorie) ?* → `treemap` ou `sunburst`.

Quelques exemples supplémentaires, sur nos données :

In [ ]:
px.histogram(df, x="volume_m3", nbins=30, title="Distribution des volumes")

In [ ]:
px.box(df, x="type_batiment", y="volume_m3", title="Volumes par type (avec valeurs atypiques)")

In [ ]:
px.treemap(df, path=["arrondissement", "type_batiment"], values="volume_m3",
          title="Volumes par arrondissement puis par type")

**Pourquoi Plotly, et pas Matplotlib (utilisé au cours 7) ?** Les deux font le même métier, mais pas pour le même usage :
- **Matplotlib** produit une **image fixe** (PNG). Plus simple, plus rapide, très bien pour un rapport imprimé ou une
  pièce jointe email.
- **Plotly** produit un objet **interactif** (survol, zoom, légende cliquable), exportable en page web autonome
  (cours 10). Idéal pour un tableau de bord qu'on explore, moins pour un document figé.

Il n'y a pas de "meilleur" outil dans l'absolu : le bon choix dépend de si le résultat sera *regardé une fois* (image)
ou *manipulé* (interactif).

## 16. Pourquoi ces outils, et pas d'autres ?

Un résumé des choix faits dans ce cours, et ce qu'ils remplacent :

- **pandas plutôt qu'Excel pour manipuler les données** : Excel devient lent et fragile au-delà de quelques dizaines
  de milliers de lignes ; pandas encaisse des millions de lignes, et le code peut se rejouer à l'identique sur un
  nouveau fichier (contrairement à des clics dans une interface, qu'il faut refaire à la main).
- **JupyterLite (navigateur) plutôt qu'une installation classique pour cet atelier** : zéro installation, zéro droit
  administrateur, fonctionne sur n'importe quel poste. La contrepartie : pas d'accès aux bases de données internes,
  et le stockage disparaît si le navigateur est vidé. D'où l'étape 12 pour celles et ceux qui veulent aller plus loin.
- **Une API plutôt qu'un fichier téléchargé à la main pour les données ouvertes** : la donnée reste à jour à chaque
  appel, et on ne rapatrie que ce dont on a besoin (`where`, `select`), plutôt qu'un fichier de plusieurs dizaines de Mo.
- **pivot_table plutôt que crosstab ou pivot par défaut** : c'est la seule à combiner plusieurs agrégations, les
  totaux et la gestion des valeurs manquantes en un seul appel — les deux autres sont des cas particuliers plus simples.
- **Plotly plutôt que Matplotlib pour le tableau de bord final** : parce que le livrable (cours 10) est fait pour être
  exploré (zoom, survol), pas juste regardé une fois.

Retenez surtout ceci : il n'existe presque jamais un outil "meilleur" dans l'absolu, seulement un outil mieux adapté
à ce qu'on veut faire *avec* le résultat ensuite.